# 05 — Writing a Custom Experiment

Every experiment in `QickworkspaceV2/experiments/` follows the same two-class pattern:

```
MyProgram(BaseProgram)      ← defines the QICK pulse sequence
MyExperiment(BaseExperiment) ← manages sweep, acquisition, analysis, saving
```

This notebook builds a **Spin Echo** experiment from scratch to illustrate the stable public pattern.
You can test a custom class directly from a notebook; registry entry is only needed after its API is stable.

In [ ]:
from qick.asm_v2 import QickSweep1D

from QickworkspaceV2 import BaseExperiment, ExperimentConfig
from QickworkspaceV2.analysis.qubit import SpinEchoAnalysis
from QickworkspaceV2.core.base_program import BaseProgram
from QickworkspaceV2.config.system_cfg import config_list

# data_path belongs to this notebook/session and may point outside the project.
BaseExperiment.connect_pyro4(
    ns_host='192.168.10.82',
    data_path=r'D:\Labber_Data\your_experiment',
)

# config_list may also be imported from any external module on sys.path.
cfg_all = ExperimentConfig(config_list)
cfg = cfg_all.get_qubit('Q1')
cfg.update({
    'steps': 101,
    'wait_time': QickSweep1D('waitloop', 0.0, 100.0),
    'virtual_detune': 0.1,
})

## Step 1 — Define the QICK program

`_initialize(cfg)` declares channels and the sweep loop.  
`_body(cfg)` is the pulse sequence executed for each loop iteration.

Key `BaseProgram` helpers you can call inside `_initialize` / `_body`:

| Method | What it does |
|---|---|
| `setup_resonator(cfg)` | Declare readout channel and pulse |
| `setup_qubit_gen(cfg, prefix)` | Declare qubit drive generator |
| `setup_standard_gates(cfg, prefix)` | Register x180, y180, x90, x90m, y90, y90m |
| `add_loop(name, n)` | Add a sweep loop of `n` steps |
| `pulse(ch, name, t)` | Fire a named pulse |
| `delay_auto(dt)` | Auto-delay between pulses |
| `measure(cfg)` | Trigger readout ADC |

In [ ]:
class SpinEchoProgram(BaseProgram):
    """π/2 — wait τ/2 — π — wait τ/2 — π/2 — readout.  Sweeps τ."""

    def _initialize(self, cfg):
        self.setup_resonator(cfg)
        self.setup_qubit_gen(cfg, prefix='ge')
        self.add_loop('waitloop', cfg['steps'])
        self.setup_qb_pulse(cfg, 'ge', name='x90_1', gain_key='pi2_gain_ge')
        self.setup_qb_pulse(cfg, 'ge', name='x180', gain_key='pi_gain_ge')
        phase = cfg.get('qb_phase', 0) + cfg['wait_time'] * 360 * cfg['virtual_detune']
        self.setup_qb_pulse(
            cfg, 'ge', name='x90_2', gain_key='pi2_gain_ge', phase=phase
        )

    def _body(self, cfg):
        self.send_readoutconfig(ch=cfg['ro_ch'], name='myro', t=0)
        self.pulse(ch=cfg['qb_ch'], name='x90_1', t=0)
        self.delay_auto(cfg['wait_time'] / 2 + 0.01, tag='wait1')
        self.pulse(ch=cfg['qb_ch'], name='x180', t=0)
        self.delay_auto(cfg['wait_time'] / 2 + 0.01, tag='wait2')
        self.pulse(ch=cfg['qb_ch'], name='x90_2', t=0)
        self.delay_auto(0.01)
        self.measure(cfg)

## Step 2 — Define the experiment class

The class attributes tell the framework how to display and save results.  
Bind an `Analysis` class so complex IQ conversion, fitting, quality checks, and plotting stay reusable.

In [ ]:
class SpinEcho(BaseExperiment):
    """Spin Echo — extracts T2 echo coherence time."""

    EXPT_NAME    = 'custom_spin_echo'
    TAG          = 'Coherence'
    X_LABEL      = 'Wait time (µs)'
    TITLE_PREFIX = 'Spin Echo'
    SWEEP_KEYS_TO_REMOVE = ['wait_time']
    X_SAVE_NAME  = 'Time'
    X_SAVE_UNIT  = 'us'
    X_SAVE_SCALE = 1.0
    Analysis = SpinEchoAnalysis

    def _create_program(self):
        return SpinEchoProgram(
            self.soccfg,
            reps=self.cfg['reps'],
            final_delay=self.cfg['relax_delay'],
            cfg=self.cfg,
        )

    def _extract_sweep_axis(self, prog):
        wait1 = prog.get_time_param('wait1', 't', as_array=True)
        wait2 = prog.get_time_param('wait2', 't', as_array=True)
        return wait1 + wait2

## Step 3 — Run it

In [ ]:
expt = SpinEcho(cfg)
result = expt.run(py_avg=800)

print('Quality     :', result.quality)
print('fit_result  :', result.fit_result)
print('T2e         :', result.get_param('T2e_us'))

In [ ]:
# The bound Analysis class owns IQ-channel selection and fit rendering.
expt.plot(analyze=False)

## Where to put the new experiment

Test the classes directly in a notebook first. No registry entry is required.

When the experiment is stable and should be available to generic clients:

1. Save the two classes in `QickworkspaceV2/experiments/coherence/spin_echo.py`
2. Export from `QickworkspaceV2/experiments/coherence/__init__.py`:
   ```python
   from .spin_echo import SpinEcho
   ```
3. Add an `ExperimentSpec` to `core/experiment_registry.py` only after `run()` reliably returns `ExperimentData`.
4. Import paths inside `QickworkspaceV2/experiments/coherence/` are **3 dots** deep:
   ```python
   from ...core.base_program    import BaseProgram
   from ...core.base_experiment import BaseExperiment
   from ...analysis.qubit       import SpinEchoAnalysis
   ```

**Never import from `qick_workspace`** — all utilities live in `QickworkspaceV2/tools/`.

**Next:** [06_real_hardware.ipynb](06_real_hardware.ipynb) — connecting to QICK hardware, saving data, and running the REST service.